In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

test = pd.read_csv(
    "../data/raw/test_FD001.txt",
    sep=r"\s+",
    header=None
)

In [2]:
columns = [
    "engine_id",
    "cycle",
    "setting_1",
    "setting_2",
    "setting_3"
]

number_of_sensor_columns = test.shape[1] - 5

sensor_columns = [
    f"sensor_{i}"
    for i in range(1, number_of_sensor_columns + 1)
]

columns += sensor_columns

test.columns = columns

In [3]:
true_rul = pd.read_csv(
    "../data/raw/RUL_FD001.txt",
    sep=r"\s+",
    header=None
)

true_rul.columns = ["RUL"]

In [5]:
test = test.sort_values(
    ["engine_id", "cycle"]
).copy()

selected_sensors = [
    "sensor_2", "sensor_3", "sensor_4", "sensor_7", "sensor_8", "sensor_9", "sensor_11", "sensor_12", "sensor_13", "sensor_14", "sensor_15", "sensor_17", "sensor_20"
]

for sensor in selected_sensors:

    test[f"{sensor}_roll5"] = (
        test.groupby("engine_id")[sensor]
        .transform(
            lambda x: x.rolling(
                window=5,
                min_periods=1
            ).mean()
        )
    )

for sensor in selected_sensors:

    test[f"{sensor}_roll10"] = (
        test.groupby("engine_id")[sensor]
        .transform(
            lambda x: x.rolling(
                window=10,
                min_periods=1
            ).mean()
        )
    )

def calculate_slope(values):

    x = np.arange(len(values))

    slope = np.polyfit(
        x,
        values,
        1
    )[0]

    return slope
    
for sensor in selected_sensors:

    test[f"{sensor}_slope10"] = (
        test.groupby("engine_id")[sensor]
        .transform(
            lambda x: x.rolling(
                window=10,
                min_periods=2
            ).apply(
                calculate_slope,
                raw=True
            )
        )
        .fillna(0)
    )


In [9]:
features_std10 = ([f"{sensor}_std10" for sensor in std_sensors])

In [ ]:
std_sensors = [
    "sensor_4",
    "sensor_11",
    "sensor_2",
    "sensor_15",
    "sensor_9"
]

for sensor in std_sensors:

    test[f"{sensor}_std10"] = (
        test.groupby("engine_id")[sensor]
        .transform(
            lambda x: x.rolling(
                window=10,
                min_periods=2
            ).std()
        )
        .fillna(0)
    )



features_final = (
    ["cycle"]
    + selected_sensors
    + [f"{sensor}_roll5" for sensor in selected_sensors]
    + [f"{sensor}_roll10" for sensor in selected_sensors]
    + [f"{sensor}_slope10" for sensor in selected_sensors]
    + [f"{sensor}_std10" for sensor in std_sensors]
)
test_last_cycle = (
    test
    .sort_values(["engine_id", "cycle"])
    .groupby("engine_id")
    .tail(1)
    .sort_values("engine_id")
    .reset_index(drop=True)
)
X_official_test = test_last_cycle[
    features_final
].copy()

In [14]:
RUL_CAP = 125

y_official_capped = true_rul["RUL"].clip(
    upper=RUL_CAP
)

print(y_official_capped.head())
print("Labels:", len(y_official_capped))
print("Test engines:", len(X_official_test))

0    112
1     98
2     69
3     82
4     91
Name: RUL, dtype: int64
Labels: 100
Test engines: 100


In [15]:
import xgboost as xgb

final_xgb_capped = xgb.XGBRegressor()

In [16]:
final_xgb_capped.load_model(
    "../models/xgboost_capped.json"
)

In [19]:
trained_features = final_xgb_capped.get_booster().feature_names

print("Model expects:")
print(trained_features)

print("\nOfficial test has:")
print(X_official_test.columns.tolist())

print("\nModel feature count:", len(trained_features))
print("Test feature count:", X_official_test.shape[1])

Model expects:
['cycle', 'sensor_2', 'sensor_3', 'sensor_4', 'sensor_7', 'sensor_8', 'sensor_9', 'sensor_11', 'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15', 'sensor_17', 'sensor_20', 'sensor_2_roll5', 'sensor_3_roll5', 'sensor_4_roll5', 'sensor_7_roll5', 'sensor_8_roll5', 'sensor_9_roll5', 'sensor_11_roll5', 'sensor_12_roll5', 'sensor_13_roll5', 'sensor_14_roll5', 'sensor_15_roll5', 'sensor_17_roll5', 'sensor_20_roll5', 'sensor_2_roll10', 'sensor_3_roll10', 'sensor_4_roll10', 'sensor_7_roll10', 'sensor_8_roll10', 'sensor_9_roll10', 'sensor_11_roll10', 'sensor_12_roll10', 'sensor_13_roll10', 'sensor_14_roll10', 'sensor_15_roll10', 'sensor_17_roll10', 'sensor_20_roll10', 'sensor_2_slope10', 'sensor_3_slope10', 'sensor_4_slope10', 'sensor_7_slope10', 'sensor_8_slope10', 'sensor_9_slope10', 'sensor_11_slope10', 'sensor_12_slope10', 'sensor_13_slope10', 'sensor_14_slope10', 'sensor_15_slope10', 'sensor_17_slope10', 'sensor_20_slope10', 'sensor_4_std10', 'sensor_11_std10', 'sensor_2_std

In [20]:
test_features = X_official_test.columns.tolist()

missing = [
    feature
    for feature in trained_features
    if feature not in test_features
]

extra = [
    feature
    for feature in test_features
    if feature not in trained_features
]

print("Missing from test:", missing)
print("Extra in test:", extra)

Missing from test: ['cycle', 'sensor_2', 'sensor_3', 'sensor_4', 'sensor_7', 'sensor_8', 'sensor_9', 'sensor_11', 'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15', 'sensor_17', 'sensor_20', 'sensor_2_roll5', 'sensor_3_roll5', 'sensor_4_roll5', 'sensor_7_roll5', 'sensor_8_roll5', 'sensor_9_roll5', 'sensor_11_roll5', 'sensor_12_roll5', 'sensor_13_roll5', 'sensor_14_roll5', 'sensor_15_roll5', 'sensor_17_roll5', 'sensor_20_roll5', 'sensor_2_roll10', 'sensor_3_roll10', 'sensor_4_roll10', 'sensor_7_roll10', 'sensor_8_roll10', 'sensor_9_roll10', 'sensor_11_roll10', 'sensor_12_roll10', 'sensor_13_roll10', 'sensor_14_roll10', 'sensor_15_roll10', 'sensor_17_roll10', 'sensor_20_roll10', 'sensor_2_slope10', 'sensor_3_slope10', 'sensor_4_slope10', 'sensor_7_slope10', 'sensor_8_slope10', 'sensor_9_slope10', 'sensor_11_slope10', 'sensor_12_slope10', 'sensor_13_slope10', 'sensor_14_slope10', 'sensor_15_slope10', 'sensor_17_slope10', 'sensor_20_slope10']
Extra in test: []


In [21]:
print("Shape:", X_official_test.shape)
print("Columns:")
print(X_official_test.columns.tolist())

Shape: (100, 5)
Columns:
['sensor_4_std10', 'sensor_11_std10', 'sensor_2_std10', 'sensor_15_std10', 'sensor_9_std10']


In [22]:
print(test.columns.tolist())

['engine_id', 'cycle', 'setting_1', 'setting_2', 'setting_3', 'sensor_1', 'sensor_2', 'sensor_3', 'sensor_4', 'sensor_5', 'sensor_6', 'sensor_7', 'sensor_8', 'sensor_9', 'sensor_10', 'sensor_11', 'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15', 'sensor_16', 'sensor_17', 'sensor_18', 'sensor_19', 'sensor_20', 'sensor_21', 'sensor_2_roll5', 'sensor_3_roll5', 'sensor_4_roll5', 'sensor_7_roll5', 'sensor_8_roll5', 'sensor_9_roll5', 'sensor_11_roll5', 'sensor_12_roll5', 'sensor_13_roll5', 'sensor_14_roll5', 'sensor_15_roll5', 'sensor_17_roll5', 'sensor_20_roll5', 'sensor_2_roll10', 'sensor_3_roll10', 'sensor_4_roll10', 'sensor_7_roll10', 'sensor_8_roll10', 'sensor_9_roll10', 'sensor_11_roll10', 'sensor_12_roll10', 'sensor_13_roll10', 'sensor_14_roll10', 'sensor_15_roll10', 'sensor_17_roll10', 'sensor_20_roll10', 'sensor_2_slope10', 'sensor_3_slope10', 'sensor_4_slope10', 'sensor_7_slope10', 'sensor_8_slope10', 'sensor_9_slope10', 'sensor_11_slope10', 'sensor_12_slope10', 'sensor_13_slope1

In [23]:
test_last_cycle = (
    test
    .sort_values(["engine_id", "cycle"])
    .groupby("engine_id")
    .tail(1)
    .sort_values("engine_id")
    .reset_index(drop=True)
)

In [24]:
X_official_test = test_last_cycle[
    trained_features
].copy()
print("Shape:", X_official_test.shape)
print("Number of expected features:", len(trained_features))

Shape: (100, 58)
Number of expected features: 58


In [25]:
official_pred = final_xgb_capped.predict(
    X_official_test
)

In [26]:
from sklearn.metrics import (
    mean_absolute_error,
    root_mean_squared_error,
    r2_score
)

official_mae = mean_absolute_error(
    y_official_capped,
    official_pred
)

official_rmse = root_mean_squared_error(
    y_official_capped,
    official_pred
)

official_r2 = r2_score(
    y_official_capped,
    official_pred
)

print("NASA Official Test MAE:", official_mae)
print("NASA Official Test RMSE:", official_rmse)
print("NASA Official Test R²:", official_r2)

NASA Official Test MAE: 12.055103302001953
NASA Official Test RMSE: 16.731138229370117
NASA Official Test R²: 0.8256824016571045
